In [394]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,r2_score,classification_report,confusion_matrix,mean_squared_error
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder,PolynomialFeatures
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier


In [395]:
data = pd.read_csv("adult.csv")
data.head()


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [396]:
data.duplicated().sum()
data.drop_duplicates(inplace=True)


In [397]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48790 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48790 non-null  int64 
 1   workclass        48790 non-null  object
 2   fnlwgt           48790 non-null  int64 
 3   education        48790 non-null  object
 4   educational-num  48790 non-null  int64 
 5   marital-status   48790 non-null  object
 6   occupation       48790 non-null  object
 7   relationship     48790 non-null  object
 8   race             48790 non-null  object
 9   gender           48790 non-null  object
 10  capital-gain     48790 non-null  int64 
 11  capital-loss     48790 non-null  int64 
 12  hours-per-week   48790 non-null  int64 
 13  native-country   48790 non-null  object
 14  income           48790 non-null  object
dtypes: int64(6), object(9)
memory usage: 6.0+ MB


In [398]:
data.workclass.unique()

array(['Private', 'Local-gov', '?', 'Self-emp-not-inc', 'Federal-gov',
       'State-gov', 'Self-emp-inc', 'Without-pay', 'Never-worked'],
      dtype=object)

In [399]:
Most_repeated_Value = data.workclass.mode()[0]

In [400]:
data.workclass = data.workclass.str.replace("?",Most_repeated_Value)

In [401]:
data.education.unique()

array(['11th', 'HS-grad', 'Assoc-acdm', 'Some-college', '10th',
       'Prof-school', '7th-8th', 'Bachelors', 'Masters', 'Doctorate',
       '5th-6th', 'Assoc-voc', '9th', '12th', '1st-4th', 'Preschool'],
      dtype=object)

In [402]:
data.education = data.education.str.strip().str.replace('-',' ')

In [403]:
education_change = str(np.random.choice(['7th','8th','1st','4th','5th','6th']))

In [404]:
data.education = data.education.replace(['7th-8th','5th-6th','1st-4th','-'],[education_change,education_change,education_change,' '])

In [405]:
data.info()


<class 'pandas.core.frame.DataFrame'>
Index: 48790 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48790 non-null  int64 
 1   workclass        48790 non-null  object
 2   fnlwgt           48790 non-null  int64 
 3   education        48790 non-null  object
 4   educational-num  48790 non-null  int64 
 5   marital-status   48790 non-null  object
 6   occupation       48790 non-null  object
 7   relationship     48790 non-null  object
 8   race             48790 non-null  object
 9   gender           48790 non-null  object
 10  capital-gain     48790 non-null  int64 
 11  capital-loss     48790 non-null  int64 
 12  hours-per-week   48790 non-null  int64 
 13  native-country   48790 non-null  object
 14  income           48790 non-null  object
dtypes: int64(6), object(9)
memory usage: 6.0+ MB


In [406]:
data['marital-status'] = data['marital-status'].str.strip().str.replace('-',' ')

In [407]:
data.occupation.unique()

array(['Machine-op-inspct', 'Farming-fishing', 'Protective-serv', '?',
       'Other-service', 'Prof-specialty', 'Craft-repair', 'Adm-clerical',
       'Exec-managerial', 'Tech-support', 'Sales', 'Priv-house-serv',
       'Transport-moving', 'Handlers-cleaners', 'Armed-Forces'],
      dtype=object)

In [408]:
data.occupation = data.occupation.str.strip().str.replace('-',' ').str.replace("?",data.occupation.mode()[0])

In [409]:
data.occupation.mode()[0]

'Prof specialty'

In [410]:
data.relationship.unique()

array(['Own-child', 'Husband', 'Not-in-family', 'Unmarried', 'Wife',
       'Other-relative'], dtype=object)

In [411]:
data.relationship = data.relationship.str.strip().str.replace("-",' ')

In [412]:
data.race.unique()

array(['Black', 'White', 'Asian-Pac-Islander', 'Other',
       'Amer-Indian-Eskimo'], dtype=object)

In [413]:
data[(data.select_dtypes("object")).replace('-',' ',regex=True).columns] = (data.select_dtypes("object")).replace('-',' ',regex=True)

In [414]:
data.head(2)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never married,Machine op inspct,Own child,Black,Male,0,0,40,United States,<=50K
1,38,Private,89814,HS grad,9,Married civ spouse,Farming fishing,Husband,White,Male,0,0,50,United States,<=50K


In [415]:
for i in data.select_dtypes("object").columns:
    uni = data[i].unique()== data[i].unique()
    print(f'[------{i}-----] :: {uni}')




[------workclass-----] :: [ True  True  True  True  True  True  True  True]
[------education-----] :: [ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True]
[------marital-status-----] :: [ True  True  True  True  True  True  True]
[------occupation-----] :: [ True  True  True  True  True  True  True  True  True  True  True  True
  True  True]
[------relationship-----] :: [ True  True  True  True  True  True]
[------race-----] :: [ True  True  True  True  True]
[------gender-----] :: [ True  True]
[------native-country-----] :: [ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True]
[------income-----] :: [ True  True]


In [416]:
data['native-country'] = data['native-country'].replace("?",data['native-country'].mode()[0])

In [417]:
data['native-country'].unique()

array(['United States', 'Peru', 'Guatemala', 'Mexico',
       'Dominican Republic', 'Ireland', 'Germany', 'Philippines',
       'Thailand', 'Haiti', 'El Salvador', 'Puerto Rico', 'Vietnam',
       'South', 'Columbia', 'Japan', 'India', 'Cambodia', 'Poland',
       'Laos', 'England', 'Cuba', 'Taiwan', 'Italy', 'Canada', 'Portugal',
       'China', 'Nicaragua', 'Honduras', 'Iran', 'Scotland', 'Jamaica',
       'Ecuador', 'Yugoslavia', 'Hungary', 'Hong', 'Greece',
       'Trinadad&Tobago', 'Outlying US(Guam USVI etc)', 'France',
       'Holand Netherlands'], dtype=object)

In [418]:
data['income'].value_counts()

income
<=50K    37109
>50K     11681
Name: count, dtype: int64

In [419]:
data.describe()

,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week
count,48790.000000,4.879000e+04,48790.000000,48790.000000,48790.000000,48790.000000
mean,38.652798,1.896690e+05,10.078807,1080.217688,87.595573,40.425886
std,13.708493,1.056172e+05,2.570046,7455.905921,403.209129,12.392729
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175550e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781385e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376062e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


In [420]:
from sklearn.preprocessing import LabelEncoder

encode = {}

for col in data.select_dtypes("object").columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encode[col] = le

In [421]:
x = data.drop('income',axis=1)

y = data.income

In [424]:
smote = SMOTE(sampling_strategy="auto",random_state=42)
X_smote,y_smote = smote.fit_resample(x,y)
data = pd.concat([X_smote,y_smote],axis=1)

In [425]:
for col in encode.keys():
    data[col] = encode[col].inverse_transform(data[col])

In [426]:
data.head(1)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never married,Machine op inspct,Own child,Black,Male,0,0,40,United States,<=50K


In [427]:
data.income.value_counts()

income
<=50K    37109
>50K     37109
Name: count, dtype: int64

In [428]:
x = data.drop('income',axis=1)

y = data.income

In [429]:
x.head(1)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never married,Machine op inspct,Own child,Black,Male,0,0,40,United States


In [430]:
data[data.income=='>50K']

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
2,28,Local gov,336951,Assoc acdm,12,Married civ spouse,Protective serv,Husband,White,Male,0,0,40,United States,>50K
3,44,Private,160323,Some college,10,Married civ spouse,Machine op inspct,Husband,Black,Male,7688,0,40,United States,>50K
7,63,Self emp not inc,104626,Prof school,15,Married civ spouse,Prof specialty,Husband,White,Male,3103,0,32,United States,>50K
10,65,Private,184454,HS grad,9,Married civ spouse,Machine op inspct,Husband,White,Male,6418,0,40,United States,>50K
14,48,Private,279724,HS grad,9,Married civ spouse,Machine op inspct,Husband,White,Male,3103,0,48,United States,>50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74213,42,Private,203233,HS grad,9,Married civ spouse,Sales,Husband,White,Male,0,0,48,United States,>50K
74214,35,Private,265556,Preschool,10,Married civ spouse,Exec managerial,Husband,Black,Male,0,0,48,Laos,>50K
74215,51,Self emp inc,181714,Prof school,15,Married civ spouse,Prof specialty,Husband,Other,Male,99999,0,62,United States,>50K
74216,55,Private,188642,9th,4,Married civ spouse,Priv house serv,Husband,White,Male,0,0,40,United States,>50K


In [431]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.20,random_state=100)


In [432]:
num_column = X_train.select_dtypes("number")
Obj_column = X_train.select_dtypes("object")
Obj_column


,workclass,education,marital-status,occupation,relationship,race,gender,native-country
46072,Private,Some college,Never married,Other service,Own child,White,Male,United States
38485,Private,HS grad,Divorced,Other service,Not in family,White,Female,United States
70266,Self emp inc,Prof school,Married civ spouse,Other service,Husband,Black,Male,United States
63434,Local gov,Masters,Married civ spouse,Armed Forces,Husband,White,Male,United States
36456,Self emp not inc,Assoc acdm,Never married,Prof specialty,Not in family,White,Female,United States
...,...,...,...,...,...,...,...,...
14260,Private,HS grad,Married civ spouse,Craft repair,Husband,White,Male,United States
63370,Private,Assoc voc,Married civ spouse,Farming fishing,Husband,White,Male,United States
65615,Private,HS grad,Married civ spouse,Sales,Other relative,Black,Female,United States
56088,Self emp inc,Bachelors,Married civ spouse,Priv house serv,Husband,White,Male,United States


In [435]:
def create_model(X_train,y_train,model):
    number_data = Pipeline(
        [
    ('scale',StandardScaler()),
            ('filling',SimpleImputer(strategy="median"))
        ]
    )
    object_data = Pipeline(
        [
            ("encode",OneHotEncoder(drop="first",sparse_output=False,handle_unknown="ignore")),
            ('filling',SimpleImputer(strategy="most_frequent"))
        ]
    )
    Preprocessing = ColumnTransformer(
        transformers=[
            ('num',number_data,make_column_selector(dtype_include=np.number)),
            ('Obj',object_data,make_column_selector(dtype_include="object"))
        ]
    )
    model_Pipeline = Pipeline(
        [
            ('Preprocessor',Preprocessing),
            ('model',model),
            
        ]
    )
    
    model_Pipeline.fit(X_train,y_train)
    return model_Pipeline


In [436]:
def Model_Predict(X_test,y_test,Model):
    y_pre = Model.predict(X_test)
    print(y_pre)
    print(f'Accuracy Score :       {accuracy_score(y_test,y_pre)}')
    print(f'Confusion matrix :     {confusion_matrix(y_test,y_pre)}')
    print(f"Classification matrix : {classification_report(y_test,y_pre)}")
    

In [437]:
Decisiontree = create_model(X_train,y_train,DecisionTreeClassifier(random_state=42))
Model_Predict(X_test,y_test,Decisiontree)


['>50K' '>50K' '<=50K' ... '>50K' '>50K' '<=50K']
Accuracy Score :       0.8592023713284829
Confusion matrix :     [[6328 1071]
 [1019 6426]]
Classification matrix :               precision    recall  f1-score   support

       <=50K       0.86      0.86      0.86      7399
        >50K       0.86      0.86      0.86      7445

    accuracy                           0.86     14844
   macro avg       0.86      0.86      0.86     14844
weighted avg       0.86      0.86      0.86     14844



In [438]:
Logistic_Regression = create_model(X_train,y_train,LogisticRegression())
Model_Predict(X_test,y_test,Logistic_Regression)


['>50K' '>50K' '<=50K' ... '<=50K' '>50K' '<=50K']
Accuracy Score :       0.8608865534896254
Confusion matrix :     [[6354 1045]
 [1020 6425]]
Classification matrix :               precision    recall  f1-score   support

       <=50K       0.86      0.86      0.86      7399
        >50K       0.86      0.86      0.86      7445

    accuracy                           0.86     14844
   macro avg       0.86      0.86      0.86     14844
weighted avg       0.86      0.86      0.86     14844



In [439]:
SVC = create_model(X_train,y_train,SVC(kernel="poly"))

In [ ]:
Model_Predict(X_test,y_test,SVC)

In [ ]:
KNN = create_model(X_train,y_train,KNeighborsClassifier(n_neighbors=3))
Model_Predict(X_test,y_test,KNN)


In [ ]:
import pickle
with open("SVC_model.pkl", "wb") as file:
    pickle.dump(SVC, file)

In [ ]:
with open("Logistic_model.pkl", "wb") as file:
    pickle.dump(Logistic_Regression ,file)

In [ ]:
with open("Decision_tree.pkl", "wb") as file:
    pickle.dump(model_Pipeline, file)

In [ ]:
with open("KNN_model.pkl", "wb") as file:
    pickle.dump(KNN, file)